# 03 — Cálculo da variável-alvo Y: `status_real`

**Projeto:** Preditor de Falhas ML (Grupo 16)  
**Disciplinas:** ED2, Redes e APS  
**Turma:** Ciência da Computação — 3.º e 4.º semestres  
**Entrega:** cálculo e documentação do rótulo `status_real`

> Este notebook documenta e demonstra o cálculo solicitado para a disciplina. A regra oficial continua na função `status_real` do pacote, em `src/preditor_de_falhas_ml/features.py`; aqui ela é aplicada aos dados locais.

## Objetivo

Recalcular e conferir a variável-alvo categórica **Y** para cada registro curated, usando perda de pacotes e latência. O contrato canônico separa X e Y nos objetos `curated/features.csv` e `curated/labels.csv` do bucket `preditor-falhas-ml`. Para executar este notebook, mantenha cópias locais em `data/curated/features.csv` e `data/curated/labels.csv`. Os arquivos são unidos 1:1 pelas chaves `msm_id`, `timestamp` e `prb_id`.

| Prioridade | Condição | Rótulo atribuído a Y |
|---:|---|---|
| 1 | `perda_pacotes_pct > 15` | `FALHA` |
| 2 | Se a primeira condição não ocorrer e `latencia_ms > 100` | `RISCO` |
| 3 | Se nenhuma das condições anteriores ocorrer | `OK` |

Os limites são estritos: valores iguais a 15% e 100 ms não ultrapassam os limites. A regra de perda tem prioridade quando as duas métricas ultrapassam seus limites. Valores ausentes seguem o comportamento da função oficial do projeto.

O escopo é recalcular `status_real` em uma cópia em memória, comparar com o rótulo armazenado em Y e resumir as classes. Os CSVs originais não são alterados, nada é gravado no S3 e nenhum modelo é treinado.

## Entrega mínima

- Explicar as condições, os limites e a prioridade dos rótulos.
- Carregar as cópias locais de `curated/features.csv` e `curated/labels.csv` e conferir suas colunas e chaves.
- Unir X e Y 1:1 pelas chaves `msm_id`, `timestamp` e `prb_id`.
- Recalcular `status_real` em uma cópia dos dados e comparar com Y armazenado.
- Exibir uma amostra, a distribuição das classes e casos de demonstração.

## 1. Identificação da equipe

| Integrante | Responsabilidade registrada no projeto |
|---|---|
| Guilherme Leite Tavares | Arquitetura de dados, quadro Kanban, especificação e revisão |
| Alexandre Tiago de Oliveira | N/A |
| Ingrid Ferreira de Sousa | N/A |
| Kauan Garcia Dias de Oliveira | Revisão do notebook / aprovação do PR |
| Lucas Eduardo Malachias Bagatela | N/A |
| Stephanie Vitoria Bessa dos Santos | Revisão / aprovação de PRs

## 2. Decisões da equipe

| Decisão | Justificativa |
|---|---|
| Usar cópias locais de `data/curated/features.csv` e `data/curated/labels.csv` | São cópias dos objetos canônicos X e Y do bucket `preditor-falhas-ml`. |
| Unir X e Y pelas chaves `msm_id`, `timestamp` e `prb_id` | Preserva o contrato 1:1 do curated S2.1. |
| Manter a regra em `status_real()` no pacote | Evita duplicar a regra de negócio no notebook e permite reutilizá-la pelo projeto. |
| Recalcular `status_real` em uma cópia em memória | Preserva os CSVs curated e permite comparar Y armazenado com a regra oficial. |
| Usar as classes `FALHA`, `RISCO` e `OK` | Segue a regra de rotulagem definida para o projeto. |
| Não treinar um modelo neste notebook | Esta entrega documenta somente o cálculo da variável-alvo Y. |

## 3. Bibliotecas e ambiente

Na raiz do repositório, execute `uv sync` e selecione no editor o kernel Python desse ambiente. O projeto já inclui `pandas` e a função de rotulagem; este notebook não adiciona dependências.

## 4. Carregar e inspecionar os dados

Baixe os objetos S3 `curated/features.csv` e `curated/labels.csv` e salve cópias locais em `data/curated/` com esses nomes. A célula abaixo localiza o repositório e carrega os dois CSVs canônicos com pandas.

In [ ]:
from pathlib import Path

import pandas as pd

from preditor_de_falhas_ml import status_real

raiz_repo = Path.cwd()
if not (raiz_repo / "data" / "curated").is_dir():
    raiz_repo = raiz_repo.parent

pasta_curated = raiz_repo / "data" / "curated"
caminho_features = pasta_curated / "features.csv"
caminho_labels = pasta_curated / "labels.csv"
for caminho in (caminho_features, caminho_labels):
    if not caminho.is_file():
        raise FileNotFoundError(
            f"Não encontrei {caminho}. "
            "Baixe curated/features.csv e curated/labels.csv do bucket "
            "preditor-falhas-ml e salve em data/curated/."
        )

features = pd.read_csv(caminho_features)
labels = pd.read_csv(caminho_labels)
print(f"Features X: {caminho_features} ({len(features):,} registros)")
display(features.head())
print(f"Labels Y: {caminho_labels} ({len(labels):,} registros)")
display(labels.head())

## 5. Conferir os schemas e unir X com Y

O arquivo de features contém as métricas e as chaves; o de labels contém as mesmas chaves e `status_real`. O notebook valida as colunas, rejeita chaves duplicadas e confere que cada linha de X tem exatamente um rótulo correspondente em Y.

In [ ]:
colunas_chave = ["msm_id", "timestamp", "prb_id"]
colunas_metricas = ["perda_pacotes_pct", "latencia_ms"]
colunas_features = [*colunas_chave, "ip", *colunas_metricas]
colunas_labels = [*colunas_chave, "status_real"]

for nome, tabela, requeridas in (
    ("features.csv", features, colunas_features),
    ("labels.csv", labels, colunas_labels),
):
    ausentes = [coluna for coluna in requeridas if coluna not in tabela.columns]
    if ausentes:
        raise ValueError(
            f"{nome} não contém as colunas necessárias: " + ", ".join(ausentes)
        )
    if tabela.duplicated(subset=colunas_chave).any():
        raise ValueError(
            f"{nome} contém chaves duplicadas em: " + ", ".join(colunas_chave)
        )

dados = features.merge(
    labels,
    on=colunas_chave,
    how="outer",
    validate="one_to_one",
    indicator=True,
)
sem_par = dados["_merge"] != "both"
if sem_par.any():
    exemplos = dados.loc[sem_par, [*colunas_chave, "_merge"]].head(5)
    raise ValueError(
        "Features e labels não têm pares 1:1 para todas as chaves. "
        f"Exemplos: {exemplos.to_dict(orient='records')}"
    )
dados = dados.drop(columns="_merge")

rotulos_originais = dados["status_real"].astype("string")
dados_rotulados = dados.copy()
for coluna in colunas_metricas:
    dados_rotulados[coluna] = pd.to_numeric(dados_rotulados[coluna], errors="raise")

print("Colunas de X validadas:", ", ".join(colunas_features))
print("Colunas de Y validadas:", ", ".join(colunas_labels))
print(f"Registros unidos por chave: {len(dados_rotulados):,}")

## 6. Calcular Y com a função do projeto

A função auxiliar converte valores ausentes do pandas para `None` e chama `status_real()`. Os limites e a prioridade ficam definidos uma única vez em `src/preditor_de_falhas_ml/features.py`. Depois do recálculo, o notebook compara o resultado com `status_real` armazenado em `labels.csv`.

In [ ]:
def rotular_linha(linha: pd.Series) -> str:
    perda = (
        None
        if pd.isna(linha["perda_pacotes_pct"])
        else float(linha["perda_pacotes_pct"])
    )
    latencia = None if pd.isna(linha["latencia_ms"]) else float(linha["latencia_ms"])
    return status_real(perda, latencia)


dados_rotulados["status_real"] = dados_rotulados.apply(rotular_linha, axis=1)
divergentes = rotulos_originais.ne(dados_rotulados["status_real"]).fillna(True)
print(f"Rótulos divergentes em relação a labels.csv: {int(divergentes.sum())}")
colunas_exibicao = [*colunas_chave, "ip", *colunas_metricas, "status_real"]
display(dados_rotulados[colunas_exibicao].head(10))

## 7. Conferir a distribuição dos rótulos

A tabela mostra a quantidade e o percentual de registros em cada classe, sempre na ordem `FALHA`, `RISCO` e `OK`.

In [ ]:
ordem_rotulos = ["FALHA", "RISCO", "OK"]
resumo = (
    dados_rotulados["status_real"]
    .value_counts()
    .reindex(ordem_rotulos, fill_value=0)
    .rename_axis("status_real")
    .to_frame("quantidade")
)
resumo["percentual_pct"] = (resumo["quantidade"] / len(dados_rotulados) * 100).round(2)

display(resumo)

## 8. Demonstrar limites, prioridade e valores ausentes

Os exemplos abaixo chamam a mesma função do projeto. Eles incluem os limites exatos, cada condição acima do limite, as duas condições verdadeiras ao mesmo tempo e um caso sem métricas disponíveis.

In [ ]:
casos_exemplo = pd.DataFrame(
    {
        "caso": [
            "Nos limites exatos",
            "Perda acima do limite",
            "Perda e latência acima dos limites",
            "Somente latência acima do limite",
            "Sem métricas disponíveis",
        ],
        "perda_pacotes_pct": [15.0, 15.1, 20.0, 10.0, pd.NA],
        "latencia_ms": [100.0, 100.0, 120.0, 100.1, pd.NA],
    }
)

casos_exemplo["status_real"] = casos_exemplo.apply(rotular_linha, axis=1)
display(casos_exemplo)

## 9. Checklist da equipe

- [x] Regras de Y, limites estritos e prioridade descritos.
- [x] Notebook chama a função oficial do pacote para criar `status_real`.
- [x] CSV original preservado; rótulo criado em uma cópia em memória.
- [x] Amostra dos dados e distribuição das classes incluídas.
- [x] Casos de limite, prioridade e ausência de métricas incluídos.
- [ ] Executar as células no ambiente do projeto e revisar os resultados antes da entrega.

## Registro final

Ao executar todas as células em sequência, o notebook carrega `curated/features.csv` e `curated/labels.csv`, confere e une X e Y pelas chaves canônicas, recalcula `status_real` em memória, informa divergências com Y armazenado e apresenta a amostra, o resumo das classes e os exemplos da regra.